In [ ]:
from ...src.RAG.data_loader import make_docs
file_path = ['../../data/stay_crawling.csv', '../../data/sports_crawling.csv', '../../data/busan_rag_data.json']
documents = make_docs(file_path)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import inspect
from rag import Rag, RagConfig

print(inspect.getsource(RagConfig))
print(inspect.signature(RagConfig))



In [1]:
from rag import Rag, RagConfig
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

DB_PATH = "./db/chroma_db"
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
ragconfig = RagConfig(db_path=DB_PATH, embeddings=embeddings)
rag = Rag(config=ragconfig)

In [2]:
db = rag.load()

[RAG-BUILDER] loading DB with path: ./db/chroma_db


In [3]:
query = "더러운 해운대의 호텔 추천해줘"
results = db.similarity_search(query, k=3)

for res in results:
    print("-" * 30)
    print(res.page_content) 

------------------------------
장소명: 해운대씨클라우드호텔레지던스
구분: 숙박업체
카테고리: 호텔
리뷰: 저렴하나 그만큼의 가치를 하는곳. 위치가 해운대에 인접해서 좋음. 방음이 안돼서 밤늦게까지 길거리 인파의 소음이 들어와 잠귀가 밝으면 못 잘듯. 예약한 방의 비데 작동 안함. 헬스장 시설 노후 및 가지수 적음. 그러나 모텔수준으로 저렴. , 뷰와 위치는 좋았습니다. 다만 숙소 위생과 방음이 아쉬웠습니다. 침구는 먼지가 많아서 재채기가 계속 나왔고 바닥도 약간 미끌거리는 느낌이 들었습니다. 또한 복도 지나다니는 소리, 야간 창밖에서 자동차 다니는 소리가 여과없이 들려서 다소 불편했습니다. 또 엘레베이터를 중간에 갈아타야 해서 시간이 많이 걸리는 점 역시 소소하게 불편했습니다. 대신 접근성과 멋진 경관이 뛰어났기에 숙소에 많이 안머무르실 분이라면 괜찮은 선택 같습니다 , 비슷한 이름의 호텔이 있어서 헷갈립니다 다른분들은 모르겠지만 저는 헷갈렸습니다. 엘리베이터를 두번 갈아타야하고 너무 작습니다. 직원분들 너무 불친절하십니다. 뭐 힘들수도 있지만 체크인때부터 뭔가 틱틱대는 말투로 안내해주는 호텔은 이곳이 처음인듯합니다. 바다앞이라는 장점을 빼고는 없는거 같습니다. 침구류나 바닥 너무 더럽습니다. 여기는다시는 머물지 않을듯 합니다.
리뷰 개수: 32개
평점: 3.2
태그: 2인실, 가족호텔, 레지던스호텔, 물놀이, 오션뷰, 조식서비스, 호캉스
가격: 룸타입슈페리어 트윈전망없음 65,000원, 룸타입18시 체크인  하프오션뷰객실 랜덤 배정 65,000원, 룸타입슈페리어 더블 하프오션 70,000원, 룸타입슈페리어 트윈 하프오션 70,000원
주소: 부산 해운대구 해운대해변로 287 6층 616호
연락처: 0519336000
출처: 카카오맵2025
------------------------------
장소명: 해운대센텀호텔
구분: 숙박업체
카테고리: 호텔
리뷰: 가격이나 객실 구조는 정말 좋은데, 청소 상태가 정말 아쉽습니다. 구석들을 많이

In [4]:
from hybrid_retriever import HybridRetriever, RetrieverConfig
BM25_PATH = "./db/bm25"
model_name = "gpt-4o"
config = RetrieverConfig(db = db, pickle_path = BM25_PATH, model_name = model_name)
retreiver = HybridRetriever(config=config)

[RETRIEVER] making dense retriever with TYPE: similarity and ARGS: {'k': 5}
[RETRIEVER] making bm25 retriever with PATH: ./db/bm25
[RETRIEVER] make chaining with the retriever
[RETRIEVER] making ensemble retriever...


In [5]:
query = "부산에서 제일 재밌는 체험활동 추천해줘봐"
retreiver.retrieve(query)

질문: 부산에서 제일 재밌는 체험활동 추천해줘봐
답변:
부산에서 재미있는 체험활동으로는 다음과 같은 장소들을 추천드립니다:

1. **자갈치크루즈**: 부산 바다의 멋진 전경과 붉은빛 석양을 감상할 수 있는 크루즈 체험입니다. 낮과 저녁 모두 다른 매력을 느낄 수 있습니다.

2. **렛츠런파크 부산경남**: 말에 대한 모든 것을 배우고 체험할 수 있는 복합 레저 공간입니다. 특히 승마 강습을 받을 수 있어 특별한 체험을 쌓기에 좋습니다.

3. **부산 아쿠아리움**: 다양한 해양 생물을 만날 수 있는 곳으로, 국내 최대 길이의 해저 터널과 상어 투명보트, VR 체험존 등 다양한 해양 생태계 액티비티를 즐길 수 있습니다.

4. **더베이101**: 요트 체험을 비롯한 해양레저와 먹거리를 즐길 수 있는 곳으로, 특히 야경이 멋진 곳으로 유명합니다.

5. **국립부산과학관**: 과학을 주제로 한 체험형 전시가 가득한 곳으로, 과학에 관심이 있는 분들에게 추천합니다.

각 장소마다 독특한 체험을 제공하니, 취향에 맞는 곳을 선택해보세요!


'부산에서 재미있는 체험활동으로는 다음과 같은 장소들을 추천드립니다:\n\n1. **자갈치크루즈**: 부산 바다의 멋진 전경과 붉은빛 석양을 감상할 수 있는 크루즈 체험입니다. 낮과 저녁 모두 다른 매력을 느낄 수 있습니다.\n\n2. **렛츠런파크 부산경남**: 말에 대한 모든 것을 배우고 체험할 수 있는 복합 레저 공간입니다. 특히 승마 강습을 받을 수 있어 특별한 체험을 쌓기에 좋습니다.\n\n3. **부산 아쿠아리움**: 다양한 해양 생물을 만날 수 있는 곳으로, 국내 최대 길이의 해저 터널과 상어 투명보트, VR 체험존 등 다양한 해양 생태계 액티비티를 즐길 수 있습니다.\n\n4. **더베이101**: 요트 체험을 비롯한 해양레저와 먹거리를 즐길 수 있는 곳으로, 특히 야경이 멋진 곳으로 유명합니다.\n\n5. **국립부산과학관**: 과학을 주제로 한 체험형 전시가 가득한 곳으로, 과학에 관심이 있는 분들에게 추천합니다.\n\n각 장소마다 독특한 체험을 제공하니, 취향에 맞는 곳을 선택해보세요!'

In [6]:
questions = ["F1963은 어떤 곳인지 좀 더 자세히 설명해줄 수 있어? 그리고 거기서 뭐 할 수 있는지 알려줘.", 
"이케아 동부산점에서 쇼핑할 때 어떤 것들이 있나요? 그리고 롯데몰 동부산점과 함께 가면 좋을까요?", 
"SOMAD 가죽공방은 어떤 체험을 제공하나요?",  
"감천문화마을은 어떤 독특한 분위기를 가지고 있나요?",
"고등어다찌 연산본점은 어떤 음식점을 제공하나요?"]
query = questions[3]
retreiver.retrieve(query)

질문: 감천문화마을은 어떤 독특한 분위기를 가지고 있나요?
답변:
감천문화마을은 산자락을 따라 늘어선 집들과 미로 같은 계단과 골목길, 그리고 알록달록 아름다운 색을 자랑하는 집들이 독특한 분위기를 만들어내며, 부산의 마추픽추로 불립니다. 또한, 계단식 주거형태와 미로 같은 골목길이 감천문화마을의 상징으로, 하늘, 산, 바다와 어우러져 환상적인 조망을 제공합니다.


'감천문화마을은 산자락을 따라 늘어선 집들과 미로 같은 계단과 골목길, 그리고 알록달록 아름다운 색을 자랑하는 집들이 독특한 분위기를 만들어내며, 부산의 마추픽추로 불립니다. 또한, 계단식 주거형태와 미로 같은 골목길이 감천문화마을의 상징으로, 하늘, 산, 바다와 어우러져 환상적인 조망을 제공합니다.'

In [7]:
from hybrid_retriever import HybridRetriever
model_name = "gpt-4o"

filter = {
    'k': 3,
    "filter": {"sort": "체험"} # 메타 데이터 필터링!
}

config = RetrieverConfig(db = db, pickle_path = BM25_PATH, model_name = model_name, search_kwargs=filter)

retreiver_filtering = HybridRetriever(config=config)


[RETRIEVER] making dense retriever with TYPE: similarity and ARGS: {'k': 3, 'filter': {'sort': '체험'}}
[RETRIEVER] making bm25 retriever with PATH: ./db/bm25
[RETRIEVER] make chaining with the retriever
[RETRIEVER] making ensemble retriever...


In [9]:
query = "부산에서 제일 재밌는 체험활동 추천해줘봐"
questions = ["F1963은 어떤 곳인지 좀 더 자세히 설명해줄 수 있어? 그리고 거기서 뭐 할 수 있는지 알려줘.", 
"이케아 동부산점에서 쇼핑할 때 어떤 것들이 있나요? 그리고 롯데몰 동부산점과 함께 가면 좋을까요?", 
"SOMAD 가죽공방은 어떤 체험을 제공하나요?",  
"감천문화마을은 어떤 독특한 분위기를 가지고 있나요?",
"고등어다찌 연산본점은 어떤 음식점을 제공하나요?"]
query = questions[3]
retreiver_filtering.retrieve(query)

질문: 감천문화마을은 어떤 독특한 분위기를 가지고 있나요?
답변:
감천문화마을은 산자락을 따라 늘어선 집들과 미로 같은 계단과 골목길, 알록달록 아름다운 색을 자랑하는 집들이 독특한 분위기를 만들어내며, 부산의 마추픽추로 불립니다.


'감천문화마을은 산자락을 따라 늘어선 집들과 미로 같은 계단과 골목길, 알록달록 아름다운 색을 자랑하는 집들이 독특한 분위기를 만들어내며, 부산의 마추픽추로 불립니다.'